# RAG Pipeline Evaluation using Ragas

This notebook evaluates the quality of our Clinical RAG Assistant. The process is as follows:
1.  **Load** a predefined set of questions and ideal answers (`ground_truth`).
2.  **Execute** our RAG pipeline for each question to generate an `answer` and retrieve `contexts`.
3.  **Evaluate** the results using the `ragas` framework to get quantitative metrics on our system's performance.

In [ ]:
!pip install ragas==0.1.7 datasets==2.19.0 pandas==2.2.2 sentence-transformers==2.7.0 chromadb-client==0.5.0 ollama==0.2.0 python-dotenv==1.0.1


import os
import logging
import pandas as pd
from datasets import Dataset
import chromadb
from sentence_transformers import SentenceTransformer
import ollama
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)
from dotenv import load_dotenv

# Configure logging to see the progress
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

## Step 1: Configuration

We define all connection parameters and model names here. For local development, these are loaded from a `.env` file located in the project's root directory. In a CI/CD or production environment, these would be set as environment variables.

In [ ]:
# Load environment variables from the root .env file
load_dotenv(dotenv_path='../.env')

# --- ChromaDB Configuration ---
CHROMA_HOST = os.getenv("CHROMA_HOST", "localhost")
CHROMA_PORT = int(os.getenv("CHROMA_PORT", "8000"))
CHROMA_COLLECTION_NAME = os.getenv("CHROMA_COLLECTION_NAME", "clinical_recommendations")

# --- Embedding Model Configuration ---
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

# --- LLM Configuration ---
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = "llama3"

# --- RAG Configuration ---
CONTEXT_RETRIEVAL_LIMIT = 3 # Number of chunks to retrieve for context

print("Configuration loaded.")

## Step 2: Load Evaluation Dataset

We load our manually curated questions and ground truth answers from the `dataset.jsonl` file. This dataset is the benchmark against which we will measure our system's performance.

In [ ]:
eval_dataset_path = '../evaluation_data/dataset.jsonl'
try:
    eval_dataset = Dataset.from_json(eval_dataset_path)
    logging.info(f"Successfully loaded evaluation dataset with {len(eval_dataset)} questions.")
    print("Dataset preview:")
    print(eval_dataset)
except Exception as e:
    logging.error(f"Failed to load dataset: {e}")

## Step 3: Replicate the RAG Pipeline

To use `ragas`, we need to generate the `answer` and `contexts` for each question. The following functions replicate the logic of our live RAG system:
1.  **Initialize Clients:** Connect to ChromaDB, load the embedding model, and connect to Ollama.
2.  **`retrieve_context`:** Takes a question, embeds it, and queries ChromaDB to find the most relevant document chunks.
3.  **`generate_answer`:** Takes the question and the retrieved context, builds a precise prompt, and gets a final answer from the LLM.

In [ ]:
# --- Initialize clients and models ---
logging.info("Initializing clients and models for evaluation...")
try:
    chroma_client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
    collection = chroma_client.get_collection(name=CHROMA_COLLECTION_NAME)
    
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    
    ollama_client = ollama.Client(host=OLLAMA_HOST)
    
    logging.info("All clients and models initialized successfully.")
except Exception as e:
    logging.error(f"Error during initialization: {e}")

In [ ]:
def retrieve_context(question: str) -> list[str]:
    """Retrieves relevant context chunks from ChromaDB for a given question."""
    if 'collection' not in locals() or collection is None:
        logging.error("ChromaDB collection is not available.")
        return []
    
    # Create an embedding for the user's question
    question_embedding = embedding_model.encode(question).tolist()
    
    # Query ChromaDB for the most similar documents
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=CONTEXT_RETRIEVAL_LIMIT,
    )
    
    return results['documents'][0] if results['documents'] else []

In [ ]:
def generate_answer(question: str, context: list[str]) -> str:
    """Generates an answer using Ollama based on the provided question and context."""
    if not context:
        return "Could not retrieve any context to answer the question."

    context_str = "\n\n".join(context)
    
    # This prompt template is crucial. It instructs the LLM to base its answer strictly on the context.
    prompt = f"""
    Based STRICTLY on the following context, please provide a concise and accurate answer to the user's question. 
    If the context does not contain the answer, state that the information is not available in the provided documents.

    CONTEXT:
    {context_str}

    QUESTION:
    {question}

    ANSWER:
    """

    try:
        response = ollama_client.chat(
            model=OLLAMA_MODEL,
            messages=[{'role': 'user', 'content': prompt}]
        )
        return response['message']['content']
    except Exception as e:
        logging.error(f"Error during Ollama generation: {e}")
        return "Error generating answer from LLM."

### Single Test Run
Before processing the whole dataset, let's test the pipeline with a single question to ensure all components are working together correctly.


In [ ]:
if 'eval_dataset' in locals() and len(eval_dataset) > 0:
    test_question = eval_dataset[0]['question']
    test_context = retrieve_context(test_question)
    test_answer = generate_answer(test_question, test_context)

    print("--- SINGLE TEST RUN ---")
    print(f"Question: {test_question}")
    print(f"\nRetrieved Contexts ({len(test_context)} chunks):")
    for i, ctx in enumerate(test_context):
        print(f"  Chunk {i+1}: {ctx[:120]}...")
    print(f"\nGenerated Answer: {test_answer}")
    print("-----------------------")

## Step 4: Run the Pipeline on the Entire Dataset

Now we will apply our RAG functions to every example in the dataset. The `.map()` function from the `datasets` library is perfect for this, as it efficiently processes each row.

In [ ]:
def generate_rag_results(example):
    """A single function to run the RAG chain for one example."""
    question = example["question"]
    contexts = retrieve_context(question)
    answer = generate_answer(question, contexts)
    
    return {
        "answer": answer,
        "contexts": contexts
    }

if 'eval_dataset' in locals():
    # This will run the RAG pipeline for each question and add the 'answer' and 'contexts' columns.
    results_dataset = eval_dataset.map(generate_rag_results)
    logging.info("Finished running RAG pipeline on the entire dataset.")
    print(results_dataset)

## Step 5: Execute Evaluation with Ragas

With our dataset now enriched with the generated answers and retrieved contexts, we can perform the final evaluation using `ragas`. We will calculate a set of key metrics that assess the performance of both our retrieval and generation steps.

In [ ]:
if 'results_dataset' in locals():
    ragas_metrics = [
        faithfulness,       # How factual is the answer based on the context?
        answer_relevancy,   # How relevant is the answer to the question?
        context_precision,  # Signal-to-noise ratio in the retrieved context.
        context_recall,     # Did we retrieve all the necessary context to answer?
    ]
    
    logging.info("Starting Ragas evaluation... This may take a few minutes.")
    
    # The evaluate function requires the LLM and embeddings to be specified for its internal calculations
    result = evaluate(
        results_dataset,
        metrics=ragas_metrics,
        llm=ollama.Bedrock(model=OLLAMA_MODEL), # Ragas needs an LLM instance for some metrics
        embeddings=embedding_model,
    )
    
    logging.info("Ragas evaluation complete.")
    
    print(result)

In [ ]:
## Step 6: Display and Interpret Results

Let's display the final scores in a clean table format for easy analysis.

---
### Cell 18 (Code):
```python
if 'result' in locals():
    evaluation_df = result.to_pandas()
    print("\n--- RAGAS EVALUATION RESULTS ---")
    display(evaluation_df)
else:
    print("\nEvaluation could not be completed.")

### Interpretation of Results

*   **`faithfulness`:** Measures if the answer is grounded in the provided context. A high score (close to 1.0) means the model is not hallucinating and is using the documents correctly. This is the most critical metric for clinical applications.
*   **`answer_relevancy`:** Measures if the answer directly addresses the question. A high score indicates the model is not generating verbose or off-topic responses.
*   **`context_precision` & `context_recall`:** These metrics evaluate the retrieval step. High recall means we are finding all the necessary information. High precision means we are not including irrelevant junk in the context, which could confuse the LLM.

A good RAG system will have high scores across all these dimensions, demonstrating both effective retrieval and faithful generation.